In [1]:
%load_ext line_profiler


In [2]:
def my_slow_function():
    a = [i**2 for i in range(10000)]
    b = sum(a)
    return b


In [3]:
%lprun -f my_slow_function my_slow_function()

Timer unit: 1e-09 s

Total time: 0.000389 s
File: /var/folders/bj/m6g82c6n41g83y3_dx02y7ch0000gp/T/ipykernel_51773/3795220269.py
Function: my_slow_function at line 1

Line #      Hits         Time  Per Hit   % Time  Line Contents
     1                                           def my_slow_function():
     2         1     362000.0 362000.0     93.1      a = [i**2 for i in range(10000)]
     3         1      26000.0  26000.0      6.7      b = sum(a)
     4         1       1000.0   1000.0      0.3      return b

In [ ]:
import geopandas as gpd
from shapely.geometry import Point
import random

# Simulate 5,000 random points 
random.seed(42)
points = [Point(random.uniform(-87, -84), random.uniform(34, 36)) for _ in range(5000)]
sightings = gpd.GeoDataFrame(geometry=points, crs="EPSG:4326")

# Load country/region polygons
counties = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
)
counties = counties[counties["CONTINENT"] == "North America"][["NAME", "geometry"]].to_crs("EPSG:4326")

def label_points(sightings, counties):
    results = []
    for i, point in enumerate(sightings.geometry):       
        for j, row in counties.iterrows():               
            if row.geometry.contains(point):
                results.append(row["NAME"])
                break
        else:
            results.append(None)
    return results



In [8]:
%lprun -f label_points label_points(sightings, counties)

Timer unit: 1e-09 s

Total time: 1.25012 s
File: /var/folders/bj/m6g82c6n41g83y3_dx02y7ch0000gp/T/ipykernel_51773/1339986449.py
Function: label_points at line 16

Line #      Hits         Time  Per Hit   % Time  Line Contents
    16                                           def label_points(sightings, counties):
    17         1       1000.0   1000.0      0.0      results = []
    18      5001   16083000.0   3216.0      1.3      for i, point in enumerate(sightings.geometry):       # ← BOTTLENECK
    19     10000  967334000.0  96733.4     77.4          for j, row in counties.iterrows():               # ← nested loop
    20     10000  225011000.0  22501.1     18.0              if row.geometry.contains(point):
    21      5000   39402000.0   7880.4      3.2                  results.append(row["NAME"])
    22      5000    2290000.0    458.0      0.2                  break
    23                                                   else:
    24                                                  

In [ ]:
sightings["county"] = label_points(sightings, counties)
print(sightings["county"].value_counts())